In [1]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

df_clean = pd.read_csv("aotizhongxin_clean.csv")

print("Dataset loaded:", df_clean.shape)
display(df_clean.head())

Saving aotizhongxin_clean.csv to aotizhongxin_clean.csv
Dataset loaded: (33902, 18)


,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station
0,1,2013,3,1,0,4.0,4.0,4.0,7.0,300.0,77.0,-0.7,1023.0,-18.8,0.0,NNW,4.4,Aotizhongxin
1,2,2013,3,1,1,8.0,8.0,4.0,7.0,300.0,77.0,-1.1,1023.2,-18.2,0.0,N,4.7,Aotizhongxin
2,3,2013,3,1,2,7.0,7.0,5.0,10.0,300.0,73.0,-1.1,1023.5,-18.2,0.0,NNW,5.6,Aotizhongxin
3,4,2013,3,1,3,6.0,6.0,11.0,11.0,300.0,72.0,-1.4,1024.5,-19.4,0.0,NW,3.1,Aotizhongxin
4,5,2013,3,1,4,3.0,3.0,12.0,12.0,300.0,72.0,-2.0,1025.2,-19.5,0.0,N,2.0,Aotizhongxin


In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [3]:
import pandas as pd

df = pd.read_csv("/content/aotizhongxin_clean.csv")

df["datetime"] = pd.to_datetime(
    df[["year", "month", "day", "hour"]]
)

df = df.set_index("datetime")

print("Dataset loaded successfully:", df.shape)
display(df.head())

Dataset loaded successfully: (33902, 18)


,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station
datetime,,,,,,,,,,,,,,,,,,
2013-03-01 00:00:00,1,2013,3,1,0,4.0,4.0,4.0,7.0,300.0,77.0,-0.7,1023.0,-18.8,0.0,NNW,4.4,Aotizhongxin
2013-03-01 01:00:00,2,2013,3,1,1,8.0,8.0,4.0,7.0,300.0,77.0,-1.1,1023.2,-18.2,0.0,N,4.7,Aotizhongxin
2013-03-01 02:00:00,3,2013,3,1,2,7.0,7.0,5.0,10.0,300.0,73.0,-1.1,1023.5,-18.2,0.0,NNW,5.6,Aotizhongxin
2013-03-01 03:00:00,4,2013,3,1,3,6.0,6.0,11.0,11.0,300.0,72.0,-1.4,1024.5,-19.4,0.0,NW,3.1,Aotizhongxin
2013-03-01 04:00:00,5,2013,3,1,4,3.0,3.0,12.0,12.0,300.0,72.0,-2.0,1025.2,-19.5,0.0,N,2.0,Aotizhongxin


In [4]:
TARGET = "PM2.5"
TARGET_NEXT = "PM2.5_next_hour"

df = df.sort_index()

df[TARGET_NEXT] = df[TARGET].shift(-1)
df = df.dropna(subset=[TARGET_NEXT])

X = df.drop(columns=[TARGET_NEXT])
y = df[TARGET_NEXT]

print("Input shape:", X.shape)
print("Target shape:", y.shape)
print("Target:", TARGET_NEXT)

Input shape: (33901, 18)
Target shape: (33901,)
Target: PM2.5_next_hour


In [5]:
# Create lag features
lag_hours = [1, 3, 6, 12, 24]

for lag in lag_hours:
    df[f"PM2.5_lag_{lag}h"] = df["PM2.5"].shift(lag)

In [6]:
rolling_windows = [3, 6, 12, 24]

previous_pm25 = df["PM2.5"].shift(1)

for window in rolling_windows:
    df[f"PM2.5_roll_mean_{window}h"] = (
        previous_pm25.rolling(window=window).mean()
    )

    df[f"PM2.5_roll_std_{window}h"] = (
        previous_pm25.rolling(window=window).std()
    )

In [7]:
df = df.sort_index()

df["PM2.5_next_hour"] = df["PM2.5"].shift(-1)

df = df.dropna(subset=["PM2.5_next_hour"])

print("Next-hour target created successfully")
print(df[["PM2.5", "PM2.5_next_hour"]].head())

Next-hour target created successfully
                     PM2.5  PM2.5_next_hour
datetime                                   
2013-03-01 00:00:00    4.0              8.0
2013-03-01 01:00:00    8.0              7.0
2013-03-01 02:00:00    7.0              6.0
2013-03-01 03:00:00    6.0              3.0
2013-03-01 04:00:00    3.0              5.0


In [8]:
df["hour"] = df.index.hour
df["dayofweek"] = df.index.dayofweek
df["month"] = df.index.month

In [9]:
# Cyclical Features
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

In [10]:
df = pd.get_dummies(
    df,
    columns=["wd"],
    drop_first=True
)

In [11]:
# Drop NA Rows
df_fe = df.dropna()

print("Rows before feature engineering:", len(df))
print("Rows after feature engineering:", len(df_fe))

Rows before feature engineering: 33900
Rows after feature engineering: 33876


In [12]:
print("Shape after dropping NaNs introduced by lagging:")
print(df_fe.shape)

df_fe.head(10)

Shape after dropping NaNs introduced by lagging:
(33876, 51)


,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,WSPM,station,PM2.5_next_hour,PM2.5_lag_1h,PM2.5_lag_3h,PM2.5_lag_6h,PM2.5_lag_12h,PM2.5_lag_24h,PM2.5_roll_mean_3h,PM2.5_roll_std_3h,PM2.5_roll_mean_6h,PM2.5_roll_std_6h,PM2.5_roll_mean_12h,PM2.5_roll_std_12h,PM2.5_roll_mean_24h,PM2.5_roll_std_24h,dayofweek,hour_sin,hour_cos,month_sin,month_cos,wd_ENE,wd_ESE,wd_N,wd_NE,wd_NNE,wd_NNW,wd_NW,wd_S,wd_SE,wd_SSE,wd_SSW,wd_SW,wd_W,wd_WNW,wd_WSW
datetime,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2013-03-02 00:00:00,25,2013,3,2,0,22.0,24.0,24.0,44.0,500.0,44.0,-0.4,1031.0,-17.6,0.0,1.4,Aotizhongxin,14.0,24.0,12.0,11.0,3.0,4.0,17.000000,6.244998,13.500000,5.612486,10.000000,5.640761,7.125000,5.041674,5,0.000000,1.000000e+00,1.0,6.123234e-17,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2013-03-02 01:00:00,26,2013,3,2,1,14.0,17.0,21.0,36.0,400.0,50.0,-1.0,1031.3,-17.3,0.0,1.1,Aotizhongxin,13.0,22.0,15.0,8.0,3.0,8.0,20.333333,4.725816,15.333333,6.377042,11.583333,6.141636,7.875000,5.833282,5,0.258819,9.659258e-01,1.0,6.123234e-17,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False
2013-03-02 02:00:00,27,2013,3,2,2,13.0,13.0,20.0,37.0,400.0,47.0,-1.5,1030.9,-16.9,0.0,1.7,Aotizhongxin,3.0,14.0,24.0,11.0,6.0,7.0,20.000000,5.291503,16.333333,5.391351,12.500000,5.535013,8.125000,5.965936,5,0.500000,8.660254e-01,1.0,6.123234e-17,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2013-03-02 03:00:00,28,2013,3,2,3,3.0,9.0,13.0,34.0,400.0,52.0,-1.4,1030.6,-17.6,0.0,1.4,Aotizhongxin,3.0,13.0,22.0,12.0,8.0,6.0,16.333333,4.932883,16.666667,5.046451,13.083333,5.142662,8.375000,6.041973,5,0.707107,7.071068e-01,1.0,6.123234e-17,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False
2013-03-02 04:00:00,29,2013,3,2,4,3.0,7.0,18.0,43.0,400.0,43.0,-1.5,1030.8,-17.7,0.0,0.9,Aotizhongxin,9.0,3.0,14.0,15.0,9.0,3.0,10.000000,6.082763,15.166667,7.467708,12.666667,5.757735,8.250000,6.123724,5,0.866025,5.000000e-01,1.0,6.123234e-17,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False
2013-03-02 05:00:00,30,2013,3,2,5,9.0,11.0,19.0,70.0,500.0,20.0,-1.8,1030.1,-17.5,0.0,2.0,Aotizhongxin,4.0,3.0,13.0,24.0,10.0,5.0,6.333333,5.773503,13.166667,8.975894,12.166667,6.336522,8.250000,6.123724,5,0.965926,2.588190e-01,1.0,6.123234e-17,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False
2013-03-02 06:00:00,31,2013,3,2,6,4.0,10.0,28.0,46.0,500.0,39.0,-2.5,1029.6,-17.7,0.0,0.7,Aotizhongxin,3.0,9.0,3.0,22.0,11.0,3.0,5.000000,3.464102,10.666667,7.284687,12.083333,6.374072,8.416667,6.085740,5,1.000000,6.123234e-17,1.0,6.123234e-17,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False
2013-03-02 07:00:00,32,2013,3,2,7,3.0,11.0,34.0,58.0,500.0,27.0,-1.7,1029.8,-17.0,0.0,1.2,Aotizhongxin,3.0,4.0,3.0,14.0,8.0,3.0,5.333333,3.214550,7.666667,5.046451,11.500000,6.789029,8.458333,6.050362,5,0.965926,-2.588190e-01,1.0,6.123234e-17,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False
2013-03-02 08:00:00,33,2013,3,2,8,3.0,7.0,21.0,49.0,500.0,43.0,-0.4,1029.6,-17.6,0.0,1.8,Aotizhongxin,10.0,3.0,9.0,13.0,11.0,3.0,5.333333,3.214550,5.833333,4.215052,11.083333,7.166314,8.458333,6.050362,5,0.866025,-5.000000e-01,1.0,6.123234e-17,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False


In [13]:
# Print Dataset Shape Change
print("Dataset shape after feature creation (before dropna):")
print(df.shape)

Dataset shape after feature creation (before dropna):
(33900, 51)


In [14]:
# Show all feature names
print("Final feature columns:\n")
for col in df.columns:
    print(col)

Final feature columns:

No
year
month
day
hour
PM2.5
PM10
SO2
NO2
CO
O3
TEMP
PRES
DEWP
RAIN
WSPM
station
PM2.5_next_hour
PM2.5_lag_1h
PM2.5_lag_3h
PM2.5_lag_6h
PM2.5_lag_12h
PM2.5_lag_24h
PM2.5_roll_mean_3h
PM2.5_roll_std_3h
PM2.5_roll_mean_6h
PM2.5_roll_std_6h
PM2.5_roll_mean_12h
PM2.5_roll_std_12h
PM2.5_roll_mean_24h
PM2.5_roll_std_24h
dayofweek
hour_sin
hour_cos
month_sin
month_cos
wd_ENE
wd_ESE
wd_N
wd_NE
wd_NNE
wd_NNW
wd_NW
wd_S
wd_SE
wd_SSE
wd_SSW
wd_SW
wd_W
wd_WNW
wd_WSW


In [15]:
df.head()

,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,WSPM,station,PM2.5_next_hour,PM2.5_lag_1h,PM2.5_lag_3h,PM2.5_lag_6h,PM2.5_lag_12h,PM2.5_lag_24h,PM2.5_roll_mean_3h,PM2.5_roll_std_3h,PM2.5_roll_mean_6h,PM2.5_roll_std_6h,PM2.5_roll_mean_12h,PM2.5_roll_std_12h,PM2.5_roll_mean_24h,PM2.5_roll_std_24h,dayofweek,hour_sin,hour_cos,month_sin,month_cos,wd_ENE,wd_ESE,wd_N,wd_NE,wd_NNE,wd_NNW,wd_NW,wd_S,wd_SE,wd_SSE,wd_SSW,wd_SW,wd_W,wd_WNW,wd_WSW
datetime,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2013-03-01 00:00:00,1,2013,3,1,0,4.0,4.0,4.0,7.0,300.0,77.0,-0.7,1023.0,-18.8,0.0,4.4,Aotizhongxin,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,0.000000,1.000000,1.0,6.123234e-17,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False
2013-03-01 01:00:00,2,2013,3,1,1,8.0,8.0,4.0,7.0,300.0,77.0,-1.1,1023.2,-18.2,0.0,4.7,Aotizhongxin,7.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,0.258819,0.965926,1.0,6.123234e-17,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False
2013-03-01 02:00:00,3,2013,3,1,2,7.0,7.0,5.0,10.0,300.0,73.0,-1.1,1023.5,-18.2,0.0,5.6,Aotizhongxin,6.0,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,0.500000,0.866025,1.0,6.123234e-17,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False
2013-03-01 03:00:00,4,2013,3,1,3,6.0,6.0,11.0,11.0,300.0,72.0,-1.4,1024.5,-19.4,0.0,3.1,Aotizhongxin,3.0,7.0,4.0,NaN,NaN,NaN,6.333333,2.081666,NaN,NaN,NaN,NaN,NaN,NaN,4,0.707107,0.707107,1.0,6.123234e-17,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False
2013-03-01 04:00:00,5,2013,3,1,4,3.0,3.0,12.0,12.0,300.0,72.0,-2.0,1025.2,-19.5,0.0,2.0,Aotizhongxin,5.0,6.0,8.0,NaN,NaN,NaN,7.000000,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,4,0.866025,0.500000,1.0,6.123234e-17,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False


In [16]:
# To make sure NaN count is zero
df_fe.isna().sum().sort_values(ascending=False)

,0
No,0
year,0
month,0
day,0
hour,0
PM2.5,0
PM10,0
SO2,0
NO2,0
CO,0


In [17]:
df_fe.describe().T

,count,mean,std,min,25%,50%,75%,max
No,33876.0,17801.297497,10102.941146,25.000000,9105.750000,1.802450e+04,26561.250000,35062.000000
year,33876.0,2014.699758,1.171510,2013.000000,2014.000000,2.015000e+03,2016.000000,2017.000000
month,33876.0,6.449404,3.423307,1.000000,3.000000,7.000000e+00,9.000000,12.000000
day,33876.0,15.577400,8.793544,1.000000,8.000000,1.500000e+01,23.000000,31.000000
hour,33876.0,11.528102,6.908375,0.000000,6.000000,1.200000e+01,18.000000,23.000000
PM2.5,33876.0,82.769096,82.376441,3.000000,22.000000,5.800000e+01,114.000000,898.000000
PM10,33876.0,110.021501,95.766939,2.000000,38.000000,8.600000e+01,154.000000,984.000000
SO2,33876.0,17.218922,22.745883,0.285600,3.000000,9.000000e+00,21.000000,341.000000
NO2,33876.0,59.201106,37.213510,2.000000,30.000000,5.300000e+01,82.000000,290.000000
CO,33876.0,1266.793414,1251.915113,100.000000,500.000000,9.000000e+02,1500.000000,10000.000000


In [18]:
from pathlib import Path

OUTPUT_PATH = Path("/content/aotizhongxin_features.csv")

df_fe.to_csv(OUTPUT_PATH, index=False)

print(f"Feature-engineered dataset saved to: {OUTPUT_PATH}")

Feature-engineered dataset saved to: /content/aotizhongxin_features.csv


In [19]:
from google.colab import files

files.download("/content/aotizhongxin_features.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>